In [ ]:
!pip install -U chronos-forecasting -q

import numpy as np
import torch
from google.colab import drive
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report
from chronos import Chronos2Pipeline

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/diss-datasets"

device = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', device)

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device,
)
print('Chronos-2 loaded')

def get_chronos2_embeddings(pipeline, X, batch_size=16):
    all_embeddings = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        context = torch.tensor(batch, dtype=torch.float32)
        embeddings, _ = pipeline.embed(context)
        for emb in embeddings:
            pooled = emb.mean(dim=1).flatten().numpy()
            all_embeddings.append(pooled)
        print(f'  {min(i+batch_size, len(X))}/{len(X)} done', flush=True)
    return np.array(all_embeddings)

def permute_channels(X, seed):
    r = np.random.default_rng(seed)
    out = X.copy()
    for w in range(out.shape[0]):
        perm = r.permutation(out.shape[1])
        out[w] = out[w, perm]
    return out

def desync_channels(X, seed):
    r = np.random.default_rng(seed)
    out = X.copy()
    for w in range(out.shape[0]):
        for c in range(out.shape[1]):
            out[w, c] = np.roll(out[w, c], r.integers(0, out.shape[2]))
    return out

def get_chronos2_channelpooled(pipeline, X, batch_size=16):
    all_embeddings = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        context = torch.tensor(batch, dtype=torch.float32)
        embeddings, _ = pipeline.embed(context)
        for emb in embeddings:
            pooled = emb.mean(dim=(0, 1)).flatten().numpy()
            all_embeddings.append(pooled)
        print(f'  {min(i+batch_size, len(X))}/{len(X)} done', flush=True)
    return np.array(all_embeddings)

In [ ]:
data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()
y_tr = data['train']['y'].astype(int)
te = data['test']
y_te = te['y'].astype(int)
file = np.asarray(te['file'])
t0 = te['t0'].astype(int)

MERGE = 30
event_id = np.full(len(y_te), -1, dtype=int)
next_ev = 0
for f in np.unique(file):
    idx = np.where((file == f) & (y_te == 1))[0]
    if len(idx) == 0:
        continue
    order = idx[np.argsort(t0[idx])]
    starts = t0[order]
    ev_local = np.zeros(len(order), dtype=int)
    for i in range(1, len(order)):
        ev_local[i] = ev_local[i-1] + (1 if (starts[i] - starts[i-1]) > MERGE else 0)
    for local, gi in zip(ev_local, order):
        event_id[gi] = next_ev + local
    next_ev += int(ev_local.max()) + 1

n_events = next_ev
neg_mask = (y_te == 0)
print("ground-truth seizure events:", n_events)

def sens_at_fpr(y_prob, fpr_targets=(0.05, 0.10, 0.20)):
    rows = []
    for thr in np.unique(y_prob):
        pred = y_prob >= thr
        det = np.array([pred[event_id == ev].any() for ev in range(n_events)])
        rows.append((thr, det.mean(), pred[neg_mask].mean()))
    rows = np.array(rows)
    out = {}
    for tgt in fpr_targets:
        ok = rows[rows[:, 2] <= tgt]
        out[tgt] = ok[:, 1].max() if len(ok) else np.nan
    return out, rows

Ztr_base = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_train_emb.npy')
Zte_base = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_test_emb.npy')

clf_base = RidgeClassifier()
clf_base.fit(Ztr_base, y_tr)
prob_base = clf_base.decision_function(Zte_base)

res_base, _ = sens_at_fpr(prob_base)
print("\nChronos-2 baseline (v2) event-level sensitivity:")
for tgt, s in res_base.items():
    print(f"  at {int(tgt*100)}% FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events})")

In [ ]:
X_tr = data['train']['X']
X_te = data['test']['X']

X_tr_p = permute_channels(X_tr, seed=0)
X_te_p = permute_channels(X_te, seed=0)

print("extracting permuted train embeddings...")
Ztr_perm = get_chronos2_embeddings(pipeline, X_tr_p)
print("extracting permuted test embeddings...")
Zte_perm = get_chronos2_embeddings(pipeline, X_te_p)

clf_perm = RidgeClassifier()
clf_perm.fit(Ztr_perm, y_tr)
prob_perm = clf_perm.decision_function(Zte_perm)

res_perm, _ = sens_at_fpr(prob_perm)
print("\nChronos-2 permuted-channels event-level sensitivity:")
for tgt, s in res_perm.items():
    print(f"  at {int(tgt*100)}% FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events})")

np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_permuted_train_emb.npy', Ztr_perm)
np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_permuted_test_emb.npy', Zte_perm)

In [ ]:
X_tr_d = desync_channels(X_tr, seed=0)
X_te_d = desync_channels(X_te, seed=0)

print("extracting desynced train embeddings...")
Ztr_desync = get_chronos2_embeddings(pipeline, X_tr_d)
print("extracting desynced test embeddings...")
Zte_desync = get_chronos2_embeddings(pipeline, X_te_d)

clf_desync = RidgeClassifier()
clf_desync.fit(Ztr_desync, y_tr)
prob_desync = clf_desync.decision_function(Zte_desync)

res_desync, _ = sens_at_fpr(prob_desync)
print("\nChronos-2 desynced-channels event-level sensitivity:")
for tgt, s in res_desync.items():
    print(f"  at {int(tgt*100)}% FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events})")

np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_desync_train_emb.npy', Ztr_desync)
np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_desync_test_emb.npy', Zte_desync)

In [ ]:
print("extracting channel-pooled train embeddings...")
Ztr_pool = get_chronos2_channelpooled(pipeline, X_tr)
print("extracting channel-pooled test embeddings...")
Zte_pool = get_chronos2_channelpooled(pipeline, X_te)

clf_pool = RidgeClassifier(alpha=10.0)
clf_pool.fit(Ztr_pool, y_tr)
prob_pool = clf_pool.decision_function(Zte_pool)

res_pool, _ = sens_at_fpr(prob_pool)
print("\nChronos-2 channel-pooled event-level sensitivity:")
for tgt, s in res_pool.items():
    print(f"  at {int(tgt*100)}% FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events})")

np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_chanpool_train_emb.npy', Ztr_pool)
np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_chanpool_test_emb.npy', Zte_pool)

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font

XLSX = "/content/drive/MyDrive/results.xlsx"
wb = load_workbook(XLSX)
bold = Font(bold=True)
sheet_name = "RQ1 event-level ablations"
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws = wb.create_sheet(sheet_name)

ws.cell(row=1, column=1, value="RQ1: Chronos-2 channel ablations, event-level sensitivity at fixed interictal FPR (CHB-MIT v2, 34 events)").font = bold
headers = ["Ablation", "Sens @5% FPR", "Sens @10% FPR", "Sens @20% FPR"]
for c, h in enumerate(headers, 1):
    ws.cell(row=3, column=c, value=h).font = bold

rows = [
    ("Baseline (unmodified)", res_base[0.05], res_base[0.10], res_base[0.20]),
    ("Permuted channel order", res_perm[0.05], res_perm[0.10], res_perm[0.20]),
    ("Desynced channel timing", res_desync[0.05], res_desync[0.10], res_desync[0.20]),
    ("Channel-pooled (collapsed)", res_pool[0.05], res_pool[0.10], res_pool[0.20]),
]

r = 4
for row in rows:
    for c, v in enumerate(row, 1):
        cell = ws.cell(row=r, column=c, value=v)
        if c > 1:
            cell.number_format = "0.000"
    r += 1

wb.save(XLSX)
print("saved")